In [1]:
import json
import os
import pandas as pd
import sqlite3

from langchain.agents import create_sql_agent, initialize_agent
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.agents.agent_types import AgentType
from langchain.sql_database import SQLDatabase
from langchain.agents.agent_toolkits import SQLDatabaseToolkit
from langchain.agents import Tool

from pydantic import BaseModel, Field, ValidationError

from duckduckgo_search import DDGS
from typing import List, Optional, Dict

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [2]:
with open("api_key.txt", "r") as file:
    groq_api_key = file.read().strip()

In [3]:
# Import Groq llm
from langchain_groq import ChatGroq

# High creativity LLM
llm_high = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.9,
    groq_api_key=groq_api_key
)

# Low creativity (deterministic) LLM
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    groq_api_key=groq_api_key
)

print("✓ LLM models initialized successfully")

✓ LLM models initialized successfully


In [4]:
# Verify LLM is working
print("\n" + "="*70)
print("TESTING LLM WITH LOW TEMPERATURE (Deterministic)")
print("="*70)

for i in range(5):
    response = llm.predict("What is artificial intelligence?")
    print(f"\nResponse {i+1}:\n{response[:200]}...")

print("\n" + "="*70)
print("TESTING LLM WITH HIGH TEMPERATURE (Creative)")
print("="*70)

for i in range(5):
    response = llm_high.predict("What is artificial intelligence?")
    print(f"\nResponse {i+1}:\n{response[:200]}...")


TESTING LLM WITH LOW TEMPERATURE (Deterministic)

Response 1:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as:

1. **Learning**: AI systems can learn from ...

Response 2:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, ...

Response 3:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that would typically require human intelligence, such as:

1. **Learning**: AI systems can learn from ...

Response 4:
Artificial intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as:

1. **Learning**: AI systems can learn from data, ...

Response 5:
Artificial intelligence (AI) refers to the development of computer sy

In [5]:
# SQL Set up
print("\n" + "="*70)
print("SETTING UP SQL DATABASE")
print("="*70)

# Check if database exists
if not os.path.exists('customer.db'):
    raise FileNotFoundError("❌ Database file 'customer_news.db' not found! Run 'create_database.py' first.")

db = SQLDatabase.from_uri("sqlite:///customer.db")
print("✓ Database loaded successfully")

# Initialize the system message
system_message = """You are an expert SQL database agent. Your role is to accurately retrieve customer information including their email addresses and interests from the customer database. Always verify the customer's email exists before fetching their details. Provide precise and complete information."""

# Initialize the toolkit with customer database and the LLM
toolkit = SQLDatabaseToolkit(db=db, llm=llm)

# Create the SQL agent with the system message
db_agent = create_sql_agent(
    llm=llm,
    toolkit=toolkit,
    verbose=False,
    system_message=SystemMessage(system_message)
)

print("✓ SQL Agent initialized successfully")


SETTING UP SQL DATABASE
✓ Database loaded successfully
✓ SQL Agent initialized successfully


In [6]:
# Test SQL Agent - Fetch all unique emails
print("\n" + "="*70)
print("TESTING SQL AGENT - Fetching all unique email IDs")
print("="*70)

query = "Fetch all the unique email_id"
output = db_agent.invoke(query)
print(output)


TESTING SQL AGENT - Fetching all unique email IDs
{'input': 'Fetch all the unique email_id', 'output': "('alice.6eb33c45-5@gmail.com', 'alice.d56e53b6-1@gmail.com', 'emma.a88fec03-c@gmail.com', 'george.5483cb53-8@gmail.com', 'hannah.4770b814-2@gmail.com', 'ian.203631a0-b@gmail.com', 'ian.aee37571-7@gmail.com', 'julia.b5a14512-c@gmail.com', 'julia.cf4d1edb-9@gmail.com', 'julia.d77d96f3-3@gmail.com')"}


In [7]:
# Test SQL Agent - Email is not in database
print("\n" + "="*70)
print("TESTING SQL AGENT - Email NOT in database")
print("="*70)

email = "notfound@example.com"
query = f"Fetch all records with email_id {email}"
output = db_agent.invoke(query)
print(output)


TESTING SQL AGENT - Email NOT in database
{'input': 'Fetch all records with email_id notfound@example.com', 'output': "I don't know"}


In [8]:
# Test SQL Agent - Email in database with interests
print("\n" + "="*70)
print("TESTING SQL AGENT - Email in database")
print("="*70)

email = "kevin.f8641860-7@gmail.com"
query = f"Fetch all the Interests with email_id {email} in a list"
output = db_agent.invoke(query)
print(output)


TESTING SQL AGENT - Email in database
{'input': 'Fetch all the Interests with email_id kevin.f8641860-7@gmail.com in a list', 'output': '["Politics", "Startups", "Travel"]'}


In [9]:
# Tool 1: Expand Search Queries
def expand_search_queries(inputs) -> list:
    """
    Expand user interests into time-sensitive search queries for breaking news.
    Accepts either a dict (with 'interests' and 'user_query') or a raw string.
    """
    # Handle dict input
    if isinstance(inputs, dict):
        interests = inputs.get("interests", [])
        user_query = inputs.get("user_query", "")
    else:
        # Handle string input fallback
        interests = [i.strip() for i in str(inputs).split(",")]
        user_query = ""

    system_prompt = """You are an expert at creating precise, time-sensitive news search queries. 
    Your task is to expand user interests into optimized search queries that will retrieve the latest and most relevant news articles.
    - Focus on current events and breaking news
    - Include temporal keywords like 'latest', 'today', 'breaking', 'recent', '2025', 'December 2025'
    - Make queries specific and targeted to get high-quality results
    - Consider the user's specific query context when generating searches
    - Each query should be concise (3-8 words) and search-engine optimized
    """

    expanded_queries = []
    for interest in interests:
        prompt = f"Generate one search query related to: '{interest}' considering the user query: '{user_query}'"
        response = llm.predict_messages(
            [
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ]
        )
        query = response.content.strip()
        if query:
            expanded_queries.append(query)

    return expanded_queries

# Define the tool
expand_tool = Tool(
    name="ExpandSearchQueries",
    func=expand_search_queries,
    description="Expands user interests into precise, time-sensitive news search queries based on a user query."
)

print("✓ Expand Search Queries tool created")

✓ Expand Search Queries tool created


In [10]:
# Tool 2: Duckduckgo Search
def ddg_search(query: str) -> str:
    results = []
    try:
        with DDGS() as ddgs:
            for r in ddgs.text(query, max_results=5):
                title = r.get("title", "")
                url = r.get("href", "")
                body = r.get("body", "")
                res = {"title": title, "url": url, "body": body}
                results.append(res)
    except Exception as e:
        print(f"⚠ DuckDuckGo search error: {e}")
        return []
    return results

ddg_search_tool = Tool(
    name="DuckDuckGoSearch",
    func=ddg_search,
    description="Searches DuckDuckGo for recent news and returns top 5 results with URLs."
)

print("✓ DuckDuckGo Search tool created")

✓ DuckDuckGo Search tool created


In [11]:
# Tool 3: Credibility Filter
def filter_with_llm(search_results: list) -> list:
    search_results = json.dumps(search_results, indent=2)
    system_prompt = """You are a news credibility and relevance assessment expert for NewsFindr platform.
    Your role is to filter search results to ensure accuracy, credibility, and trustworthiness.
    
    Evaluation criteria:
    - Prioritize established news organizations (Reuters, BBC, CNN, The Guardian, NYTimes, etc.)
    - Include reputable industry publications and official sources
    - Filter out spam, clickbait, unreliable blogs, and low-quality sources
    - Ensure content is recent and relevant to user interests
    - Check domain reputation and editorial standards
    - Remove duplicate or very similar articles
    - Select only the most credible and informative sources
    
    Return ONLY the filtered list of credible news results in JSON format with title, url, and body."""

    prompt = "Evaluate these search results for credibility and relevance. Filter and return only trustworthy, high-quality news sources that match the user's interests:\n\n" + search_results

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )
    return response.content

credibility_tool = Tool(
    name="CredibilityFilter",
    func=filter_with_llm,
    description="Filters search results by evaluating URL credibility using LLM."
)

print("✓ Credibility Filter tool created")

✓ Credibility Filter tool created


In [12]:
# Tool 4: Summarize News
def summarize_news(url_list) -> dict:
    """
    Summarizes key news points from a list of URLs and returns both the summary and the URLs.
    """
    system_prompt = """You are a professional news summarizer for the NewsFindr platform.
    Your task is to create comprehensive yet concise summaries of news articles.
    
    Guidelines:
    - Extract and highlight the most important information and key takeaways
    - Maintain objectivity and accuracy
    - Structure summaries with clear main points
    - Include relevant dates, facts, and context
    - Keep language clear and accessible
    - Focus on what matters most to the reader
    - Provide actionable insights when relevant
    - Each summary should be informative enough to give readers a complete understanding without reading the full article
    """

    prompt = "Create a comprehensive summary of the latest news from these URLs. For each article, provide key points, main events, and important details:\n\n" + "\n".join(url_list)

    response = llm.predict_messages(
        [
            SystemMessage(content=system_prompt),
            HumanMessage(content=prompt)
        ]
    )

    summary_text = response.content
    return {"summary": summary_text, "urls": url_list}

summarize_tool = Tool(
    name="SummarizeNews",
    func=summarize_news,
    description="Generates summaries from news URLs and returns both the summary and the links."
)

print("✓ Summarize News tool created")

✓ Summarize News tool created


In [13]:
# Initialize Agent With All Tools
print("\n" + "="*70)
print("INITIALIZING MAIN AGENT")
print("="*70)

tools = [expand_tool, ddg_search_tool, credibility_tool, summarize_tool]

agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    verbose=True
)

print("✓ Main Agent initialized with all tools")


INITIALIZING MAIN AGENT
✓ Main Agent initialized with all tools


In [14]:
# Test Single Query
print("\n" + "="*70)
print("TESTING SINGLE QUERY - Alice")
print("="*70)

email = "alice.6eb33c45-5@gmail.com"
interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

# Normalize to list
if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
    interests = interest_result["output"]["items"]
elif isinstance(interest_result["output"], str):
    interests = [i.strip() for i in interest_result["output"].split(",")]
else:
    interests = interest_result["output"]

# Define the user query
user_query = "latest developments in artificial intelligence and machine learning for business"

# Agent prompt
agent_prompt = f"""
The user's interest is: {interests} and specifically looking for {user_query}.
Here is the process to follow:
1. Expand the user's interest into news search queries using the 'ExpandSearchQueries' tool.
   The input to this tool should be a dictionary like: {{'interests': {interests}, 'user_query': '{user_query}'}}.
2. Use the query generated in step 1 to search for recent news using the 'DuckDuckGoSearch' tool for all the queries.
3. Pass the combined results of all queries into 'CredibilityFilter' for filtering.
4. From the filtered results, select the top 3 latest news articles for summarization.
5. Summarize all credible results into a detailed summary using the 'SummarizeNews'.
6. Provide the final summary to the user.
"""

response = agent.run(agent_prompt)
print("\n======= FINAL RESPONSE =======")
print(response)


TESTING SINGLE QUERY - Alice


> Entering new AgentExecutor chain...
Thought: The user is looking for the latest developments in artificial intelligence and machine learning for business, and their interests include politics, technology, and business. To provide them with relevant news, I need to expand their interests into precise news search queries. This can be achieved by using the 'ExpandSearchQueries' tool, which requires a dictionary with the user's interests and their specific query.

Action: ExpandSearchQueries
Action Input: {'interests': ['["Politics"', '"Technology"', '"Business"]'], 'user_query': 'latest developments in artificial intelligence and machine learning for business'}
Observation: ['"latest politics news today"', '"Latest tech news 2025"', '"Latest business news 2025"', '"2025 AI business technology updates"']
Thought:Thought: Now that I have the expanded search queries, I need to use them to search for recent news using the 'DuckDuckGoSearch' tool. This will pr

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': "Latest News: Today's big news headlines from India and around …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at …'}, {'title': "India News | Today's latest updates and breaking news from …", 'url': 'https://www.ndtv.com/india', 'body': "2 days ago · India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, …"}, {'title': 'Latest Entertainment News | Bollywood News | Hollywood News …', 'url': 'https://www.ndtv.com/ent

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': "Latest News: Today's big news headlines from India and around the …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at NDTV.com.'}, {'title': "India News | Today's latest updates and breaking news from India, Live ...", 'url': 'https://www.ndtv.com/india', 'body': "2 days ago · India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and Bollywood."}, {'title': 'Latest Entertainment News | Bollywood News | 

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': "Latest News: Today's big news headlines from India and around the …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at NDTV.com.'}, {'title': "India News | Today's latest updates and breaking news from India, …", 'url': 'https://www.ndtv.com/india', 'body': "2 days ago · India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and …"}, {'title': 'Latest Entertainment News | Bollywood News | Hollywood News –

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: []
Thought:Thought: I have obtained the search results for all the queries. However, the last query, '"2025 AI business technology updates"', returned an empty list. I will combine the results from all the queries and pass them into the 'CredibilityFilter' for filtering.

Action: CredibilityFilter
Action Input: [{'title': "Latest News: Today's big news headlines from India and around …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at NDTV.com.'}, {'title': "India News | Today's latest updates and breaking news from India, Live ...", 'ur

In [15]:
# Query Response Function
def query_response(email: str, user_query: str) -> str:
    """
    Main function to process user queries based on their email and specific request.
    """
    print(f"\n{'='*70}")
    print(f"Processing query for: {email}")
    print(f"User Query: {user_query}")
    print(f"{'='*70}\n")
    
    # Fetch interests based on the provided email
    interest_result = db_agent.invoke(f"Fetch all the Interests with email_id {email} in a list")

    # Normalize to list
    if isinstance(interest_result["output"], dict) and "items" in interest_result["output"]:
        interests = interest_result["output"]["items"]
    elif isinstance(interest_result["output"], str):
        interests = [i.strip() for i in interest_result["output"].split(",")]
    else:
        interests = interest_result["output"]

    # Agent prompt
    agent_prompt = f"""
    The user's interest is: {interests} and specifically looking for {user_query}.
    Here is the process to follow:
    1. Expand the user's interest into news search queries using the 'ExpandSearchQueries' tool.
       The input to this tool should be a dictionary like: {{'interests': {interests}, 'user_query': '{user_query}'}}.
    2. Use the query generated in step 1 to search for recent news using the 'DuckDuckGoSearch' tool for all the queries.
    3. Pass the combined results of all queries into 'CredibilityFilter' for filtering.
    4. From the filtered results, select the top 3 latest news articles for summarization.
    5. Summarize all credible results into a detailed summary using the 'SummarizeNews'.
    6. Provide the final summary to the user.
    """
    
    response = agent.run(agent_prompt)
    print("\n======= FINAL RESPONSE =======")
    print(response)
    print(f"\n{'='*70}\n")
    
    return response

In [16]:
# Query 1: Julia - Automobile and Business News in India

print("\n" + "="*70)
print("QUERY 1: JULIA - AUTOMOBILE & BUSINESS NEWS IN INDIA")
print("="*70)

email = "julia.d77d96f3-3@gmail.com"
user_query = "recent automobile launches and electric vehicle news in India and business updates"
response = query_response(email, user_query)


QUERY 1: JULIA - AUTOMOBILE & BUSINESS NEWS IN INDIA

Processing query for: julia.d77d96f3-3@gmail.com
User Query: recent automobile launches and electric vehicle news in India and business updates



> Entering new AgentExecutor chain...
Thought: The user is looking for recent automobile launches and electric vehicle news in India, as well as business updates. To provide the most accurate and relevant information, I need to first expand the user's interest into precise, time-sensitive news search queries. This can be achieved by using the 'ExpandSearchQueries' tool.

Action: ExpandSearchQueries
Action Input: {'interests': ['"India"', '"Automobile"', '"Business"'], 'user_query': 'recent automobile launches and electric vehicle news in India and business updates'}
Observation: ['"India latest news today"', '"Latest automobile news 2025"', '"Latest business news 2025"', '"India latest electric vehicle launches 2025"']
Thought:Thought: Now that I have the expanded search queries, I can u

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: []
Thought:Since the observation is an empty list, it seems that the search query '"India latest news today"' did not yield any results. I will try the next query.

Action: DuckDuckGoSearch
Action Input: '"Latest automobile news 2025"'

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': "Latest News: Today's big news headlines from India and around …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at …'}, {'title': "India News | Today's latest updates and breaking news from …", 'url': 'https://www.ndtv.com/india', 'body': "2 days ago · India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, …"}, {'title': 'Latest Entertainment News | Bollywood News | Hollywood News …', 'url': 'https://www.ndtv.com/ent

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: []
Thought:Since the observation is an empty list, it seems that the search query '"Latest business news 2025"' did not yield any results. I will try the next query.

Action: DuckDuckGoSearch
Action Input: '"India latest electric vehicle launches 2025"'

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': 'India - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/India', 'body': 'India, officially the Republic of India, [j][20] is a country in South Asia.'}, {'title': 'India | History, Maps, Population, Economy, & Facts | Britannica', 'url': 'https://www.britannica.com/place/India', 'body': '4 days ago · India is a country that occupies the greater part of South Asia. It is a constitutional republic that represents a highly diverse population consisting of thousands of ethnic groups. It is made up of …'}, {'title': 'Home | National Portal of India', 'url': 'https://www.india.gov.in/', 'body': 'India Portal is developed under the Digital India Programme to provide citizens with open and easy access to information. It is designed, developed, and maintained by the National Informatics Centre (NIC), …'}, {'title': 'India - The World Factbook', 'url': 'https://www.cia.gov/the-world-factbook/countries/india/', 'body': '5 days ago · Visit the Definitions and Notes page

In [17]:
# Query 2: George - Politics and Science in India

print("\n" + "="*70)
print("QUERY 2: GEORGE - POLITICS & SCIENCE IN INDIA")
print("="*70)

email = "george.5483cb53-8@gmail.com"
user_query = "latest political developments in India and breakthrough scientific discoveries"
response = query_response(email, user_query)


QUERY 2: GEORGE - POLITICS & SCIENCE IN INDIA

Processing query for: george.5483cb53-8@gmail.com
User Query: latest political developments in India and breakthrough scientific discoveries



> Entering new AgentExecutor chain...
To address the user's interest in the latest political developments in India and breakthrough scientific discoveries, we first need to expand their interests into precise, time-sensitive news search queries. This involves using the 'ExpandSearchQueries' tool with a dictionary that includes both their interests and the specific query they have.

Action: ExpandSearchQueries
Action Input: {'interests': ['"India"', '"Politics"', '"Science"'], 'user_query': 'latest political developments in India and breakthrough scientific discoveries'}
Observation: ['"India latest news today"', '"Latest politics news today"', '"Latest science news 2025"', '"India latest political news 2025"']
Thought:Now that we have the expanded search queries, we need to use these queries to se

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': 'India News, Latest Breaking News Headlines from India ... - India Today', 'url': 'https://www.indiatoday.in/india', 'body': 'India News Today: Read latest and breaking news headlines from India. Top India News, Current headlines, videos, photos and live india news updates at India Today'}, {'title': 'India Today - Latest News, Breaking News Today - Entertainment, …', 'url': 'https://www.indiatoday.in/', 'body': 'Check out the latest news from India and around the world. Latest India news on Bollywood, Politics, Business, Cricket, Technology and Travel.'}, {'title': 'Latest News Today: Current News, Big News Headlines - India Today', 'url': 'https://www.indiatoday.in/latest-news', 'body': "Today's Latest News: Read Latest Big news headlines from India and around the world with live coverage on India Today."}, {'title': 'Latest Headlines, Breaking News, Top Video News - India Today', 'url': 'https://www.indiatoday.in/news.html', 'body': 'Latest News Headline - S

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: []
Thought:Action: DuckDuckGoSearch
Action Input: '"Latest science news 2025"'

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': "Latest News: Today's big news headlines from India and around the …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at NDTV.com.'}, {'title': "India News | Today's latest updates and breaking news from India, …", 'url': 'https://www.ndtv.com/india', 'body': "2 days ago · India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and …"}, {'title': 'Latest Entertainment News | Bollywood News | Hollywood News –

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': 'India - Wikipedia', 'url': 'https://en.wikipedia.org/wiki/India', 'body': 'India, officially the Republic of India, [j][20] is a country in South Asia.'}, {'title': 'India | History, Maps, Population, Economy, & Facts | Britannica', 'url': 'https://www.britannica.com/place/India', 'body': '4 days ago · India is a country that occupies the greater part of South Asia. It is a constitutional republic that represents a highly diverse population consisting of thousands of ethnic groups. It …'}, {'title': 'Home | National Portal of India', 'url': 'https://www.india.gov.in/', 'body': 'India Portal is developed under the Digital India Programme to provide citizens with open and easy access to information. It is designed, developed, and maintained by the National …'}, {'title': 'India - The World Factbook', 'url': 'https://www.cia.gov/the-world-factbook/countries/india/', 'body': '5 days ago · Visit the Definitions and Notes page to view a description of each topic.'},

In [18]:
# Query 3: Hannah - Entertainment and Technology Business

print("\n" + "="*70)
print("QUERY 3: HANNAH - ENTERTAINMENT & TECH BUSINESS")
print("="*70)

email = "hannah.4770b814-2@gmail.com"
user_query = "trending entertainment news and technology innovations in business"
response = query_response(email, user_query)


QUERY 3: HANNAH - ENTERTAINMENT & TECH BUSINESS

Processing query for: hannah.4770b814-2@gmail.com
User Query: trending entertainment news and technology innovations in business



> Entering new AgentExecutor chain...
To address the user's interest in trending entertainment news and technology innovations in business, we first need to expand their interests into precise, time-sensitive news search queries. This involves using the 'ExpandSearchQueries' tool, which requires a dictionary with the user's interests and a specific query as inputs.

Action: ExpandSearchQueries
Action Input: {'interests': ['"Entertainment"', '"Business"', '"Technology"'], 'user_query': 'trending entertainment news and technology innovations in business'}
Observation: ['"Latest Entertainment News 2025"', '"latest business news today"', '"Latest tech news 2025"', '"Latest tech trends in entertainment 2025"']
Thought:Now that we have the expanded search queries, we can use them to search for recent news using t

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': "Latest News: Today's big news headlines from India and around …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at …'}, {'title': "India News | Today's latest updates and breaking news from …", 'url': 'https://www.ndtv.com/india', 'body': "2 days ago · India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, …"}, {'title': 'Latest Entertainment News | Bollywood News | Hollywood News …', 'url': 'https://www.ndtv.com/ent

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': 'Latest News Today: Latest Business News India, Latest …', 'url': 'https://www.businesstoday.in/latest', 'body': "Latest News India: Check today's top business news, current and latest news updates on business, economy, corporate, share market, latest stock market news headlines on Business...Business today offering the latest …Economy News India Today: Get latest …"}, {'title': 'Business News Live, Share Market News - Read Latest Finance …', 'url': 'https://economictimes.indiatimes.com/', 'body': 'Budget 2026-27 is expected to further accelerate this development. Lemon Tree Hotels and Fleur Hotels are undergoing a significant restructuring, with Warburg Pincus investing up to Rs 960 …'}, {'title': 'Business News, Finance News, India News, Stock Markets …', 'url': 'https://www.business-standard.com/', 'body': 'Business News: Get latest news on share market, personal finance news, economy news, stock market updates, company news, election results, politics news,

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': "Latest News: Today's big news headlines from India and around the …", 'url': 'https://www.ndtv.com/latest', 'body': '17 hours ago · Latest news headlines from India & around the world. Check out today’s news coverage live with videos & photos on NDTV.com.'}, {'title': "Today's Top News Stories, Latest News Today - NDTV.com", 'url': 'https://www.ndtv.com/top-stories', 'body': '14 hours ago · Top News Stories of the day, Latest News Headlines, News Specials, Breaking News and Latest India News, World current affairs & Political News all around the clock at NDTV.com.'}, {'title': "India News | Today's latest updates and breaking news from India, …", 'url': 'https://www.ndtv.com/india', 'body': "2 days ago · India News | Latest India News | Read latest and breaking news from India. Today's top India news headlines, news on Indian politics, elections, government, business, technology, and …"}, {'title': 'Latest Entertainment News | Bollywood News | Hollywood News –

/var/folders/b0/glhd5m4s5d93thmlrwhz3x1r0000gn/T/ipykernel_1895/1555600105.py:5: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: [{'title': '町田市の観光スポット完全ガイド｜家族・デート・一人旅にお ...', 'url': 'https://machida-ozawa-takeru.hatenablog.com/entry/2025/12/11/221420', 'body': 'Dec 11, 2025 · 家族連れ、カップル、一人旅など、どんな旅行スタイルでも楽しめる多彩なスポットが揃っています。 この記事では、町田市の魅力的な観光スポットを徹底解説します。'}, {'title': '「お子さま連れでも安心！ファミリーおすすめプラン」町田の ...', 'url': 'https://www.rurubu.travel/familyplan/area/japan/machida', 'body': '「お子さま連れでも安心！ ファミリーおすすめプラン」町田の宿・ホテル・旅館。 るるぶトラベルなら人気宿のお得な期間限定プランが満載。 当日／直前のオンライン予約も26時まで「お子さま連 …'}, {'title': '【2026最新】町田で家族におすすめのスポットTOP13', 'url': 'https://aumo.jp/areas/159/scenes/32', 'body': 'Jan 1, 2026 · おでかけ情報サービス「aumo」では、町田で家族におすすめの人気スポットを掲載。 実際に利用したユーザーの口コミ、写真などの情報が満載です。 「町田」「家族」に関するス …'}, {'title': '町田市で楽しめるホテル・旅館 子供の遊び場・お出かけ ...', 'url': 'https://iko-yo.net/facilities?city_ids[]=313032&genre_ids[]=40&prefecture_ids[]=13', 'body': '町田市にある子供が喜ぶ、親子で楽しめるホテル・旅館をご紹介します。 キッズ向けのメニューやキャラクタールームがあるホテルなど、ファミリーにうれしい設備＆サービスが充実。 安心して家 …'}, {'title': '町田の家族旅行におすすめのホテル・旅館 【Yahoo!トラベル】', 'url': 'https://travel.yahoo.co.jp/tokyo/1403120